# Keyword-Category Mapping

Turns LLM-validated category-category pairs into final keyword-level mappings
Handles both Category and OPC variants.

### Install packages and dependencies

In [ ]:
import os
import re
import time
from typing import List

import boto3
import numpy as np
import pandas as pd
from tqdm import tqdm

import sys
_REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

from utils.secrets import fetch_secret
from utils.text_utils import preprocess, non_alpha_1_word
from config import (
    CLIENT_PIPELINE_TYPE, TRANSFORM_PARAMS,
    FLAG_WEIGHT, ELIGIBLE_WEIGHT,
    SKU_ID_CONSTANT, SOURCE_CONSTANT,
)

In [ ]:
def download_s3_file(s3_input_path, local_input_path, access_key, secret_key):
    s3 = boto3.client("s3", aws_access_key_id=access_key, aws_secret_access_key=secret_key, region_name="us-west-2")
    local_input_path = local_input_path + s3_input_path.split("/")[-1]
    bucket_name = s3_input_path.split("/")[2]
    s3_input_path = "/".join(s3_input_path.split("/")[3:])
    print("Downloading the file >>")
    s3.download_file(bucket_name, s3_input_path, local_input_path)
    print("Download file - Local input path: ", local_input_path)
    return local_input_path


def download_s3_dir(s3_input_path, s3_dir_name, local_input_path, access_key, secret_key):
    local_paths = []
    s3 = boto3.client('s3', aws_access_key_id=access_key, aws_secret_access_key=secret_key, region_name='us-west-2')
    bucket_name = s3_input_path.split("/")[2]
    s3_input_path = "/".join(s3_input_path.split("/")[3:])
    s3_objects = s3.list_objects(Bucket=bucket_name, MaxKeys=123, Prefix=s3_input_path + s3_dir_name)
    s3_files = s3_objects["Contents"]
    for s3_object in s3_files:
        path, filename = os.path.split(s3_object["Key"])
        local_input_file_name = local_input_path + s3_dir_name + filename
        s3.download_file(bucket_name, f"{path}/{filename}", local_input_file_name)
        local_paths.append(local_input_file_name)
    return local_paths


def upload_s3_file(local_output_path, s3_output_path, access_key, secret_key):
    s3 = boto3.client("s3", aws_access_key_id=access_key, aws_secret_access_key=secret_key, region_name="us-west-2")
    bucket_name = s3_output_path.split("/")[2]
    s3_output_path = "/".join(s3_output_path.split("/")[3:])
    s3.upload_file(local_output_path, bucket_name, s3_output_path)
    print("Uploaded file to s3: " + s3_output_path)
    print("Uploaded file from: " + local_output_path)

### Gender detection

In [ ]:
def detect_keyword_gender(keyword):
    if not isinstance(keyword, str):
        return 'unisex'
    kw = keyword.lower()
    women_patterns = r'\b(women|womens|women\'s|female|ladies|girls?|womenswear|women\'s wear|women wear|female fashion|female clothing|for her|\bher\b|mom|sister|aunt|grandmother|girlfriend)\b'
    men_patterns = r'\b(men|mens|men\'s|male|gents|boys?|menswear|men\'s wear|men wear|male fashion|male clothing|for him|\bhis\b|dad|brother|uncle|grandfather|boyfriend)\b'
    has_women = bool(re.search(women_patterns, kw))
    has_men = bool(re.search(men_patterns, kw))
    if has_women and has_men:
        return 'unisex'
    if has_women:
        return 'women'
    if has_men:
        return 'men'
    return 'unisex'

def detect_category_gender(category):
    cat = category.lower()
    women_patterns = r'(for women|women\'s|\bwomens?\b|female|ladies|girls?|womenswear|women\'s wear|women wear|female fashion|female clothing|for her|\bher\b|mom|sister|aunt|grandmother|girlfriend)'
    men_patterns = r'(for men|men\'s|\bmens?\b|male|gents|boys?|menswear|men\'s wear|men wear|male fashion|male clothing|for him|\bhis\b|dad|brother|uncle|grandfather|boyfriend)'
    has_women = bool(re.search(women_patterns, cat))
    has_men = bool(re.search(men_patterns, cat))
    if has_women and has_men:
        return 'neutral'
    if has_women:
        return 'women'
    if has_men:
        return 'men'
    return 'neutral'


def apply_gender_filter(df, kw_col):
    print("6c. Applying gender filter---------------")
    before = len(df)
    df['kw_gender'] = df[kw_col].apply(detect_keyword_gender)
    df['catg_gender'] = df['similar_category'].apply(detect_category_gender)
    mask = (
        (df['kw_gender'] == 'unisex')
        | (df['kw_gender'] == df['catg_gender'])
        | (df['catg_gender'] == 'neutral')
    )
    filtered = df[mask].copy()
    removed = before - len(filtered)
    print(f"  Keyword gender distribution: {df['kw_gender'].value_counts().to_dict()}")
    if before > 0:
        print(f"  Removed {removed} rows due to gender mismatch ({removed / before * 100:.1f}%)")
    else:
        print("  No rows to filter (empty input)")
    filtered.drop(columns=['kw_gender', 'catg_gender'], inplace=True)
    return filtered

### Blacklist helpers

In [ ]:
def load_kw_blacklist_file(blacklist_loc):
    if os.path.exists(blacklist_loc):
        bl = pd.read_csv(blacklist_loc, sep="\t", encoding="utf-8")
        for col in ['keyword', 'category']:
            bl[col] = (bl[col].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True).str.lower())
        bl = bl.loc[bl['keyword'].ne('') & bl['category'].ne('')].drop_duplicates()
        print(f"  Keyword-level blacklist loaded: {len(bl)} pairs from {blacklist_loc}")
        return bl
    print(f"  No blacklist file found at {blacklist_loc}")
    return pd.DataFrame(columns=['keyword', 'category'])


def load_kw_blacklist_from_athena(client_id):
    print("  Loading keyword-level blacklist from Athena...")
    ATHENA_DATABASE = "search_relevancy_db"
    ATHENA_TABLE = "blacklist_keyword_category_pairs"
    ATHENA_OUTPUT_S3 = "s3://os-reporting-prod-buckets/similar_catg_opc_output_transformations/athena-query-results/"
    ATHENA_REGION = "us-east-1"

    query = f"""
        SELECT keyword, category
        FROM {ATHENA_DATABASE}.{ATHENA_TABLE}
        WHERE marketplace_client_id = {client_id}
    """
    athena_client = boto3.client("athena", region_name=ATHENA_REGION)
    response = athena_client.start_query_execution(
        QueryString=query,
        QueryExecutionContext={"Database": ATHENA_DATABASE},
        ResultConfiguration={"OutputLocation": ATHENA_OUTPUT_S3},
    )
    query_execution_id = response["QueryExecutionId"]

    while True:
        status = athena_client.get_query_execution(QueryExecutionId=query_execution_id)
        state = status["QueryExecution"]["Status"]["State"]
        if state in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        time.sleep(1)

    if state != "SUCCEEDED":
        reason = status["QueryExecution"]["Status"].get("StateChangeReason", "Unknown")
        print(f"  Athena query failed: {reason}. Skipping blacklist.")
        return pd.DataFrame(columns=['keyword', 'category'])

    result_s3_path = f"{ATHENA_OUTPUT_S3}{query_execution_id}.csv"
    bl = pd.read_csv(result_s3_path)
    if bl.empty:
        print("  No blacklist pairs found in Athena.")
        return pd.DataFrame(columns=['keyword', 'category'])

    for col in ['keyword', 'category']:
        bl[col] = (bl[col].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True).str.lower())
    bl = bl.loc[bl['keyword'].ne('') & bl['category'].ne('')].drop_duplicates()
    print(f"  Keyword-level blacklist loaded from Athena: {len(bl)} pairs")
    return bl


def apply_kw_blacklist_to_output(df, blacklist_df, kw_col):
    if blacklist_df is None or blacklist_df.empty:
        return df
    before = len(df)
    df = pd.merge(
        df,
        blacklist_df[['keyword', 'category']].assign(_bl=1),
        how='left',
        left_on=[kw_col, 'similar_category'],
        right_on=['keyword', 'category'],
    )
    df = df[df['_bl'] != 1].drop(columns=['_bl', 'category'], errors='ignore')
    if 'keyword_x' in df.columns:
        df = df.rename(columns={'keyword_x': 'keyword'})
    if 'keyword_y' in df.columns:
        df = df.drop(columns=['keyword_y'], errors='ignore')
    after = len(df)
    if before - after > 0:
        print(f"  Keyword-level blacklist (final output): removed {before - after} rows")
    return df

## Data Manipulation

### 1. Keyword to category mapping file

In [ ]:
def key_catg_map_manipulation(key_catg_map_loc, catg_cols, kw_col, blacklist_df=None):
    print("1. Keyword to category calculations---------------")
    key_catg_map = pd.read_csv(key_catg_map_loc, sep=',', encoding='utf-8')
    key_catg_map[catg_cols] = key_catg_map[catg_cols].fillna('').apply(lambda col: col.str.lower())

    tqdm.pandas(desc="Creating not null category list")
    key_catg_map['not_null_catg'] = key_catg_map[catg_cols].progress_apply(
        lambda row: [col for col in row if col != ''], axis=1
    )
    tqdm.pandas(desc="Creating not null categories concat")
    key_catg_map['concat'] = key_catg_map['not_null_catg'].progress_apply(lambda x: ">".join(x))

    # Apply keyword-level blacklist before dominant selection
    if blacklist_df is not None and not blacklist_df.empty:
        before = len(key_catg_map)
        key_catg_map = pd.merge(
            key_catg_map,
            blacklist_df[['keyword', 'category']].assign(_bl=1),
            how='left',
            left_on=[kw_col, 'concat'],
            right_on=['keyword', 'category'],
        )
        key_catg_map = key_catg_map[key_catg_map['_bl'] != 1].drop(columns=['_bl', 'category'], errors='ignore')
        if 'keyword_x' in key_catg_map.columns:
            key_catg_map = key_catg_map.rename(columns={'keyword_x': 'keyword'})
        if 'keyword_y' in key_catg_map.columns:
            key_catg_map = key_catg_map.drop(columns=['keyword_y'], errors='ignore')
        after = len(key_catg_map)
        print(f"  Keyword-level blacklist: removed {before - after} rows")

    print("Shape of the keyword to category mapping df: ", key_catg_map.shape)
    print(f"Unique keywords for which we have category mapping: {key_catg_map[kw_col].nunique()}")

    key_catg_map.sort_values(by=[kw_col, "score"], ascending=[False, False], inplace=True)
    key_catg_map["cumsum"] = key_catg_map.groupby(kw_col)["score"].cumsum()
    key_catg_map["total"] = key_catg_map.groupby(kw_col)["score"].transform("sum")
    key_catg_map["roll_contri"] = round(key_catg_map["cumsum"] / key_catg_map["total"], 2)
    key_catg_map["contri"] = round(key_catg_map["score"] / key_catg_map["total"], 2)
    key_catg_map["rnk"] = key_catg_map.groupby(kw_col)["score"].rank(method="first", ascending=False)

    tqdm.pandas(desc="Flagging the dominant categories")
    key_catg_map["is_dominant"] = key_catg_map.progress_apply(
        lambda x: 1 if (x["contri"] >= 0.25 and x["rnk"] == 1) else 0, axis=1,
    )

    # Gender-aware dominant reassignment
    print("  Checking keyword vs dominant category gender alignment...")
    key_catg_map['kw_gender'] = key_catg_map[kw_col].apply(detect_keyword_gender)
    key_catg_map['catg_gender'] = key_catg_map['concat'].apply(detect_category_gender)

    dom_rows = key_catg_map[key_catg_map['is_dominant'] == 1].copy()
    mismatch_kws = dom_rows[
        (dom_rows['kw_gender'] != 'unisex') &
        (dom_rows['catg_gender'] != 'neutral') &
        (dom_rows['kw_gender'] != dom_rows['catg_gender'])
    ][kw_col].unique()

    n_reassigned = 0
    if len(mismatch_kws) > 0:
        print(f"  Gender mismatches found for {len(mismatch_kws)} keywords. Reassigning dominant...")
        for kw in mismatch_kws:
            kw_mask = key_catg_map[kw_col] == kw
            kw_gender = key_catg_map.loc[kw_mask, 'kw_gender'].iloc[0]
            compatible = key_catg_map.loc[
                kw_mask & ((key_catg_map['catg_gender'] == kw_gender) | (key_catg_map['catg_gender'] == 'neutral'))
            ]
            key_catg_map.loc[kw_mask, 'is_dominant'] = 0
            if len(compatible) > 0:
                best_idx = compatible['score'].idxmax()
                key_catg_map.loc[best_idx, 'is_dominant'] = 1
                n_reassigned += 1
            else:
                orig_dom = key_catg_map.loc[kw_mask & (key_catg_map['rnk'] == 1)]
                if len(orig_dom) > 0:
                    key_catg_map.loc[orig_dom.index[0], 'is_dominant'] = 1
        print(f"  Reassigned dominant category for {n_reassigned}/{len(mismatch_kws)} mismatched keywords")
    else:
        print("  No gender mismatches found.")

    key_catg_map.drop(columns=['kw_gender', 'catg_gender'], inplace=True)

    contri_threshold = 0.90
    prev_roll_contri = key_catg_map.groupby(kw_col)["roll_contri"].shift(1, fill_value=0)
    key_catg_map["is_eligible"] = (
        (key_catg_map["roll_contri"] <= contri_threshold) | (
            (key_catg_map["roll_contri"] > contri_threshold) & (prev_roll_contri < contri_threshold)
        )
    ).astype(int)

    _base_cols = ['keyword', 'score', 'count', 'concat']
    if 'actual_keyword' in key_catg_map.columns and kw_col == 'actual_keyword':
        _base_cols = ['keyword', 'actual_keyword', 'score', 'count', 'concat']
    if 'source' in key_catg_map.columns:
        _base_cols.append('source')

    key_catg_map_v2 = key_catg_map.loc[key_catg_map["is_dominant"] == 1, _base_cols]
    key_catg_map_non_dom = key_catg_map.loc[
        (key_catg_map["is_dominant"] == 0) & (key_catg_map["is_eligible"] == 1), _base_cols
    ]

    print("Shape of the final keyword to category dataset: ", key_catg_map_v2.shape)
    print(f"Unique selected keywords with category mapping: {key_catg_map_v2[kw_col].nunique()}")
    print("Shape of the non-dominant keyword to category dataset: ", key_catg_map_non_dom.shape)

    return key_catg_map_v2, key_catg_map_non_dom

### 2. Targeted keywords

In [ ]:
def targeted_kw_manipulation(targ_key_loc):
    print("2. Targeted keywords calculations---------------")
    headers = [
        'marketplace_client_id', 'marketing_campaign_id', 'campaign_id', 'match_type',
        'keyword', 'bidding_value', 'status', 'is_negative',
    ]
    dtype_mapping = {'bidding_value': str, 'is_negative': str}
    targ_key_df = pd.read_csv(
        targ_key_loc, sep="\t", encoding='utf-8', names=headers, dtype=dtype_mapping, on_bad_lines='skip',
    )
    targ_key_df["lower_keyword"] = targ_key_df["keyword"].str.lower().str.strip()
    targ_key_df = targ_key_df.loc[targ_key_df['is_negative'] == '0']
    targ_key_df_v2 = targ_key_df[["lower_keyword"]].drop_duplicates()
    print("Targeted keyword: ", targ_key_df_v2.shape)
    return targ_key_df_v2

### 3. Filtering keywords from request-response file based on set word count

In [ ]:
def req_res_manipulation(req_res_key_loc, word_cnt_max):
    print("3. Keyword level request-response data calculations---------------")
    headers = [
        'marketplace_client_id', 'filter_page_type', 'keywords', 'trimmed_keywords',
        'word_cnt', 'request_cnt', 'response_cnt',
    ]
    res_req_df = pd.read_csv(req_res_key_loc, sep="\t", encoding='utf-8', names=headers)
    res_req_df_v2 = res_req_df[res_req_df["word_cnt"] <= word_cnt_max]
    res_req_df_v3 = res_req_df_v2[["keywords"]].drop_duplicates().reset_index(drop=True)
    res_req_df_v3.rename(columns={"keywords": "lower_keyword"}, inplace=True)
    print("Request response data: ", res_req_df_v3.shape)
    return res_req_df_v3

In [ ]:
def single_word_dataset(targ_key_df, res_req_df):
    print("4. Single word dataset calculations---------------")
    key_list_fn = pd.concat([targ_key_df, res_req_df], axis=0)
    key_list_fn_v2 = key_list_fn.drop_duplicates()
    tqdm.pandas(desc="Flagging the unwanted keywords")
    key_list_fn_v2["start_flag"] = key_list_fn_v2.progress_apply(non_alpha_1_word, axis=1)
    key_list_fn_v3 = key_list_fn_v2.loc[key_list_fn_v2["start_flag"] == 1].copy()
    key_list_fn_v3.drop(columns=["start_flag"], inplace=True)
    print("Single word keywords data: ", key_list_fn_v3.shape)
    return key_list_fn_v3

In [ ]:
def selected_keyword_dominant_catg(key_catg_map, key_list_df, kw_col, kw_group_cols):
    print("5. Selected keyword dominant catg calculations---------------")
    key_catg_map_v3 = pd.merge(key_catg_map, key_list_df, how="inner", left_on=kw_col, right_on="lower_keyword")

    print("Shape of the dataset with selected keywords categories: ", key_catg_map_v3.shape)
    print(f"Unique selected keywords with categories: {key_catg_map_v3[kw_col].nunique()}")

    key_catg_map_v3['score_adj'] = key_catg_map_v3['score'] * 0.7
    key_catg_map_v3['count_adj'] = key_catg_map_v3['count'] * 0.7
    key_catg_map_v3['score_min'] = key_catg_map_v3.groupby(kw_group_cols)['score_adj'].transform('min')
    key_catg_map_v3['count_min'] = key_catg_map_v3.groupby(kw_group_cols)['count_adj'].transform('min')
    key_catg_map_v3 = key_catg_map_v3.drop(columns=['score', 'count', 'score_adj', 'count_adj'])
    key_catg_map_v3 = key_catg_map_v3.rename(columns={'score_min': 'score', 'count_min': 'count'})
    return key_catg_map_v3

### Joining keywords to similar categories and producing scored, ranked output

- Taking category-to-category similarity pairs &  filtering out weak/irrelevant ones
- Matching keywords to their similar categories & adjusting scores based on relevance and eligibility
- Applying gender filtering
- Finally producing a ranked output file.

In [ ]:
def cat_catg_model_output(catg_catg_mod_out_loc, min_score_threshold, flag_weight):
    print("6. Model output calculation---------------")
    catg_catg_mod_out = pd.read_csv(catg_catg_mod_out_loc, sep='\t', encoding='utf-8')
    catg_catg_mod_out_v2 = catg_catg_mod_out[
        (catg_catg_mod_out["similarity_score"] >= min_score_threshold)
        & (catg_catg_mod_out["llm_flag"] != "IRRELEVANT")
        & (catg_catg_mod_out["category_concat"] != catg_catg_mod_out["similar_category"])
    ].copy()
    catg_catg_mod_out_v2["flag_weight"] = catg_catg_mod_out_v2["llm_flag"].map(flag_weight)
    print("Model output shape - Category-Category mapping: ", catg_catg_mod_out_v2.shape)
    print(f"Model output - Unique categories: {catg_catg_mod_out_v2.category_concat.nunique()}")
    return catg_catg_mod_out_v2


def merge_dominant_catg_similar_catg(
    catg_cols, key_catg_map, catg_catg_map, non_dom_catg_map,
    eligible_weight, kw_col, kw_group_cols, pipeline_type,
):
    print("7. Final calculations---------------")

    # Build output column list based on pipeline type
    if pipeline_type == "opc":
        output_cols = ["keyword"] + catg_cols + [
            "count", "score", "source",
            "keyword_opc_status", "category_opc_status",
            "tag", "opc_score", "advertisable_sku_count",
            "is_dominant", "is_eligible", "llm_flag", "final_rnk",
        ]
    else:
        output_cols = [
            "keyword", "actual_keyword", "sku_id",
        ] + catg_cols + [
            "count", "score", "source",
        ]

    catg_catg_mod_out_v3 = pd.merge(
        key_catg_map,
        catg_catg_map[['category_concat', 'similar_category', 'similarity_score', 'flag_weight', 'llm_flag']],
        how="inner",
        left_on="concat",
        right_on="category_concat",
    )

    catg_catg_mod_out_v3["rnk"] = catg_catg_mod_out_v3.groupby(
        [kw_col, "similar_category"]
    )["score"].rank(method="first", ascending=False)
    catg_catg_mod_out_v4 = catg_catg_mod_out_v3[catg_catg_mod_out_v3["rnk"] == 1].copy()

    catg_catg_mod_out_v4["source"] = SOURCE_CONSTANT
    if pipeline_type == "opc":
        catg_catg_mod_out_v4["keyword_opc_status"] = True
        catg_catg_mod_out_v4["category_opc_status"] = True
        catg_catg_mod_out_v4["tag"] = "advertisable_skus_present"
        catg_catg_mod_out_v4["opc_score"] = 10
        catg_catg_mod_out_v4["advertisable_sku_count"] = 10
        catg_catg_mod_out_v4["is_dominant"] = 0
    else:
        catg_catg_mod_out_v4["sku_id"] = SKU_ID_CONSTANT

    # is_eligible via left-join sentinel
    non_dom_lookup = non_dom_catg_map[[kw_col, 'concat']].drop_duplicates()
    non_dom_lookup['_overlap_key'] = 1
    catg_catg_mod_out_v4 = pd.merge(
        catg_catg_mod_out_v4, non_dom_lookup, how="left",
        left_on=[kw_col, "similar_category"], right_on=[kw_col, "concat"],
        suffixes=("", "_non_dom"),
    )
    catg_catg_mod_out_v4["is_eligible"] = catg_catg_mod_out_v4["_overlap_key"].fillna(0).astype(int)
    catg_catg_mod_out_v4.drop(columns=["concat_non_dom", "_overlap_key"], inplace=True, errors='ignore')

    catg_catg_mod_out_v4["eligible_weight"] = catg_catg_mod_out_v4["is_eligible"].map(eligible_weight)

    catg_catg_mod_out_v4['score'] = catg_catg_mod_out_v4['score'] * catg_catg_mod_out_v4['similarity_score'] * catg_catg_mod_out_v4['flag_weight'] * catg_catg_mod_out_v4['eligible_weight']
    catg_catg_mod_out_v4['count'] = catg_catg_mod_out_v4['count'] * catg_catg_mod_out_v4['similarity_score'] * catg_catg_mod_out_v4['flag_weight'] * catg_catg_mod_out_v4['eligible_weight']

    catg_catg_mod_out_v4 = apply_gender_filter(catg_catg_mod_out_v4, kw_col)

    # Split similar_category path back into columns
    split_cols = catg_catg_mod_out_v4['similar_category'].str.split('>', expand=True).fillna('')
    catg_df = pd.DataFrame(columns=catg_cols)
    for col, category in zip(catg_df.columns, split_cols.columns):
        catg_df[col] = split_cols[category]
    catg_catg_mod_out_v5 = pd.concat(
        [catg_catg_mod_out_v4.reset_index(drop=True), catg_df.reset_index(drop=True)], axis=1,
    ).fillna('')

    if pipeline_type == "opc":
        combined = catg_catg_mod_out_v5[output_cols[:-1]].copy()
        combined.drop_duplicates(inplace=True)
        combined.sort_values(by=["keyword", "is_eligible", "score"], ascending=[True, False, False], inplace=True)
        combined["final_rnk"] = combined.groupby("keyword").cumcount() + 1
        combined = combined[output_cols]
        print(f"Output shape: {combined.shape}")
        print(f"Eligible rows (is_eligible=1): {(combined['is_eligible']==1).sum()}")
        print(f"Non-eligible rows (is_eligible=0): {(combined['is_eligible']==0).sum()}")
    else:
        combined = catg_catg_mod_out_v5[output_cols].copy()
        combined.drop_duplicates(inplace=True)
        combined.sort_values(by=["actual_keyword", "score"], ascending=[True, False], inplace=True)
        print(f"Output shape: {combined.shape}")

    return combined

## Run pipeline

In [ ]:
def run(client_id):
    pipeline_type = CLIENT_PIPELINE_TYPE.get(str(client_id), "catg")
    params = TRANSFORM_PARAMS[pipeline_type]
    print(f"Pipeline type: {pipeline_type}")

    kw_col = params["kw_join_col"]
    kw_group_cols = params["kw_group_cols"]
    catg_cols = params["catg_cols"]
    min_score_threshold = params["min_score_threshold"]
    word_cnt_max = params["word_cnt_max"]
    output_file = params["output_file_template"].format(client_id=client_id)

    s3_key_catg_input_path = params["s3_key_catg_input"].format(client_id=client_id)
    s3_input_base = params["s3_input_base"].format(client_id=client_id)
    s3_tar_key_input_path = f"{s3_input_base}/targeted_keyword_abstract_{client_id}.tsv"
    s3_req_res_input_path = f"{s3_input_base}/keywords_req_res_cnt_{client_id}_combined_data"
    s3_model_out_path = params["s3_model_output"].format(client_id=client_id)
    s3_output_path = params["s3_output"].format(client_id=client_id)

    # S3 credentials
    access_details = fetch_secret(name="category_category_similarity", as_json=True)
    access_key = access_details['ACCESS_KEY']
    secret_key = access_details['SECRET_KEY']

    # Download inputs
    _dl_kwargs = dict(local_input_path="/tmp/", access_key=access_key, secret_key=secret_key)
    key_catg_map_loc = download_s3_file(s3_input_path=s3_key_catg_input_path, **_dl_kwargs)
    targ_key_loc = download_s3_file(s3_input_path=s3_tar_key_input_path, **_dl_kwargs)
    req_res_key_loc = download_s3_file(s3_input_path=s3_req_res_input_path, **_dl_kwargs)
    catg_catg_mod_out_loc = download_s3_file(s3_input_path=s3_model_out_path, **_dl_kwargs)

    # Load blacklist
    if params["blacklist_source"] == "athena":
        blacklist_df = load_kw_blacklist_from_athena(client_id)
    else:
        blacklist_loc = f"/home/jupyter/ds-search-relevancy-repo/input_data/{client_id}/blacklist_keyword_category_{client_id}.tsv"
        blacklist_df = load_kw_blacklist_file(blacklist_loc)

    # Run pipeline
    key_catg_map_v2, key_catg_map_non_dom = key_catg_map_manipulation(
        key_catg_map_loc, catg_cols, kw_col, blacklist_df=blacklist_df
    )

    targ_key_df_v2 = targeted_kw_manipulation(targ_key_loc)
    res_req_df_v3 = req_res_manipulation(req_res_key_loc, word_cnt_max)
    key_list_fn_v3 = single_word_dataset(targ_key_df=targ_key_df_v2, res_req_df=res_req_df_v3)

    key_catg_map_v3 = selected_keyword_dominant_catg(
        key_catg_map=key_catg_map_v2, key_list_df=key_list_fn_v3,
        kw_col=kw_col, kw_group_cols=kw_group_cols,
    )

    non_dom_catg_filtered = pd.merge(
        key_catg_map_non_dom, key_list_fn_v3[['lower_keyword']],
        how="inner", left_on=kw_col, right_on="lower_keyword",
    )
    non_dom_catg_filtered.drop(columns=['lower_keyword'], inplace=True)
    print(f"Non-dominant categories (filtered to eligible keywords) shape: {non_dom_catg_filtered.shape}")

    catg_catg_mod_out_v2 = cat_catg_model_output(catg_catg_mod_out_loc, min_score_threshold, FLAG_WEIGHT)

    catg_catg_mod_out_v6 = merge_dominant_catg_similar_catg(
        catg_cols,
        key_catg_map=key_catg_map_v3,
        catg_catg_map=catg_catg_mod_out_v2,
        non_dom_catg_map=non_dom_catg_filtered,
        eligible_weight=ELIGIBLE_WEIGHT,
        kw_col=kw_col,
        kw_group_cols=kw_group_cols,
        pipeline_type=pipeline_type,
    )

    catg_catg_mod_out_v6 = apply_kw_blacklist_to_output(catg_catg_mod_out_v6, blacklist_df, kw_col=kw_col)

    print("Final output: ", catg_catg_mod_out_v6.shape)

    tmp_output = f"/tmp/{output_file}"
    catg_catg_mod_out_v6.to_csv(tmp_output, index=False, sep="\t")
    upload_s3_file(local_output_path=tmp_output, s3_output_path=s3_output_path,
                   access_key=access_key, secret_key=secret_key)


if __name__ == "__main__":
    run()

In [ ]:
run(client_id='163519')